# Installation

## Platform Support

Pre-built wheels are available for the following platforms and architectures:

| Platform | Architecture | AVX Support | Python Versions | Notes |
|----------|-------------|-------------|-----------------|-------|
| **Linux** | x86_64 | ✅ Yes | 3.8 - 3.13 | Intel/AMD 64-bit |
| **Linux** | aarch64 | ❌ No | 3.8 - 3.13 | ARM64 (Raspberry Pi, AWS Graviton) |
| **macOS** | x86_64 | ✅ Yes | 3.8 - 3.13 | Intel Macs |
| **macOS** | arm64 | ❌ No | 3.8 - 3.13 | Apple Silicon (M1/M2/M3/M4) |
| **Windows** | AMD64 | ✅ Yes | 3.8 - 3.13 | Intel/AMD 64-bit |
| **Windows** | ARM64 | ❌ No | 3.8 - 3.13 | Windows on ARM (Surface Pro X, etc.) |

**Notes:**
- AVX (Advanced Vector Extensions) provides optimized SIMD operations for faster computation on x86_64 processors
- ARM architectures use standard optimizations but do not support AVX instructions
- All platforms use manylinux_2_28 (Linux), macOS 10.14+, or Windows 10+ as minimum requirements
- 32-bit platforms are not supported

``tttrlib`` can either be installed from prebuilt binaries or from the source code.
For most users it is recommended to install ``tttrlib`` using the prebuilt binaries.
Below the installation using prebuilt binaries and the prerequisites to compile
``tttrlib`` are briefly outlined.


## Prebuilt binaries

It is recommended to install ``tttrlib`` for Python environments via ``conda``:

```bash
conda install -c tpeulen tttrlib
```

Alternatively, ``tttrlib`` can be installed via ``pip``:

```bash
pip install tttrlib
```

**Note:** pip will automatically select the appropriate wheel for your platform and architecture.


## Building Wheels with cibuildwheel

The project uses [cibuildwheel](https://cibuildwheel.readthedocs.io/) to build wheels for multiple Python versions and platforms. Configuration is in `pyproject.toml`.

### Prerequisites

- Python 3.8 or later
- [cibuildwheel](https://cibuildwheel.readthedocs.io/): `pip install cibuildwheel`
- Git with submodules initialized: `git clone --recursive`
- **For Linux builds:** Docker Desktop (with WSL 2 backend on Windows)
- **For cross-architecture builds:** QEMU emulation (see below)

### Local Development Build

To build and install the package locally for development:

```bash
git clone --recursive https://github.com/fluorescence-tools/tttrlib.git
cd tttrlib
pip install -e .
```

This creates an editable installation that reflects code changes without reinstalling.

### Quick Start: Build All Platforms

Build wheels for all configured platforms and architectures:

```bash
pip install cibuildwheel
python -m cibuildwheel --output-dir wheelhouse
```

This builds for all Python versions (3.8-3.13) on all platforms (Linux x86_64/aarch64, Windows AMD64/ARM64, macOS x86_64/arm64).

### Build for Specific Platform

Build all architectures for a single platform:

**Linux (x86_64 and aarch64):**
```bash
python -m cibuildwheel --platform linux --output-dir wheelhouse
```

**Windows (AMD64 and ARM64):**
```bash
python -m cibuildwheel --platform windows --output-dir wheelhouse
```

**macOS (x86_64 and arm64):**
```bash
python -m cibuildwheel --platform macos --output-dir wheelhouse
```

### Build for Specific Architecture

For more control, use environment variables to target specific architectures.

#### Prerequisites for Cross-Architecture Builds

**Windows (building Linux ARM64):**
- Docker Desktop installed and running (with WSL 2 backend)
- Ensure `python`, `pip`, and `docker` all work in `cmd.exe`

**Linux/macOS:**
- Docker installed and running
- QEMU for ARM emulation (handled automatically by cibuildwheel)

#### Step 1: Enable QEMU Emulation (Required for ARM64 on x86_64)

**On Windows (one-time per reboot):**

Before building ARM64 wheels, enable QEMU emulation in Docker:

```cmd
docker run --rm --privileged multiarch/qemu-user-static --reset -p yes
```

**On Linux:**

```bash
docker run --rm --privileged multiarch/qemu-user-static --reset -p yes
```

**On macOS:**

QEMU is handled automatically by Docker Desktop.

#### Step 2: Build for Target Architecture

Use environment variables to specify the target platform and architecture:

**Linux ARM64 (aarch64):**

Windows (cmd.exe):
```cmd
set CIBW_PLATFORM=linux
set CIBW_ARCHS_LINUX=aarch64
set CIBW_BUILD_VERBOSITY=1
python -m cibuildwheel --output-dir wheelhouse
```

Windows (PowerShell):
```powershell
$env:CIBW_PLATFORM="linux"
$env:CIBW_ARCHS_LINUX="aarch64"
$env:CIBW_BUILD_VERBOSITY="1"
python -m cibuildwheel --output-dir wheelhouse
```

Linux/macOS (bash):
```bash
export CIBW_PLATFORM=linux
export CIBW_ARCHS_LINUX=aarch64
export CIBW_BUILD_VERBOSITY=1
python -m cibuildwheel --output-dir wheelhouse
```

**Linux x86_64:**

```bash
export CIBW_PLATFORM=linux
export CIBW_ARCHS_LINUX=x86_64
python -m cibuildwheel --output-dir wheelhouse
```

**Windows ARM64:**

```cmd
set CIBW_PLATFORM=windows
set CIBW_ARCHS_WINDOWS=ARM64
python -m cibuildwheel --output-dir wheelhouse
```

*Requires Visual Studio 2022 with ARM64 build tools installed.*

**Windows AMD64:**

```cmd
set CIBW_PLATFORM=windows
set CIBW_ARCHS_WINDOWS=AMD64
python -m cibuildwheel --output-dir wheelhouse
```

**macOS ARM64 (Apple Silicon):**

```bash
export CIBW_PLATFORM=macos
export CIBW_ARCHS_MACOS=arm64
python -m cibuildwheel --output-dir wheelhouse
```

*Best results on Apple Silicon hardware. Cross-compilation from x86_64 is possible but slower.*

**macOS x86_64 (Intel):**

```bash
export CIBW_PLATFORM=macos
export CIBW_ARCHS_MACOS=x86_64
python -m cibuildwheel --output-dir wheelhouse
```

### Building Multiple Architectures

Build multiple architectures in one command by separating them with spaces:

**Linux (both x86_64 and aarch64):**
```bash
export CIBW_PLATFORM=linux
export CIBW_ARCHS_LINUX="x86_64 aarch64"
python -m cibuildwheel --output-dir wheelhouse
```

**Windows (both AMD64 and ARM64):**
```cmd
set CIBW_PLATFORM=windows
set CIBW_ARCHS_WINDOWS=AMD64 ARM64
python -m cibuildwheel --output-dir wheelhouse
```

**macOS (both x86_64 and arm64):**
```bash
export CIBW_PLATFORM=macos
export CIBW_ARCHS_MACOS="x86_64 arm64"
python -m cibuildwheel --output-dir wheelhouse
```

### Building Specific Python Versions

Limit builds to specific Python versions using `CIBW_BUILD`:

**Python 3.11 and 3.12 only:**
```bash
export CIBW_BUILD="cp311-* cp312-*"
python -m cibuildwheel --output-dir wheelhouse
```

**Single Python version (3.12) on Linux ARM64:**
```bash
export CIBW_PLATFORM=linux
export CIBW_ARCHS_LINUX=aarch64
export CIBW_BUILD="cp312-*"
python -m cibuildwheel --output-dir wheelhouse
```

**Exclude specific versions:**
```bash
export CIBW_SKIP="cp38-* cp39-*"  # Skip Python 3.8 and 3.9
python -m cibuildwheel --output-dir wheelhouse
```

### Building a Single Wheel (Current Python Only)

To build a wheel for your current Python version only (not using cibuildwheel):

```bash
pip install build
python -m build --wheel
```

The wheel will be created in the `dist/` directory.

### Verifying Built Wheels

After building, check the wheel filenames to verify the correct architecture:

**Linux/macOS:**
```bash
ls wheelhouse/
```

**Windows:**
```cmd
dir wheelhouse\
```

Expected filename patterns:
- Linux x86_64: `tttrlib-*-manylinux_2_28_x86_64.whl`
- Linux ARM64: `tttrlib-*-manylinux_2_28_aarch64.whl`
- Windows AMD64: `tttrlib-*-win_amd64.whl`
- Windows ARM64: `tttrlib-*-win_arm64.whl`
- macOS x86_64: `tttrlib-*-macosx_*_x86_64.whl`
- macOS ARM64: `tttrlib-*-macosx_*_arm64.whl`

### Testing Built Wheels

After building, test the wheel before uploading:

```bash
pip install wheelhouse/tttrlib-*.whl
python -c "import tttrlib; d = tttrlib.TTTR(); print(type(d).__name__)"
```

Expected output: `TTTR`

### Troubleshooting cibuildwheel

**QEMU not registered (Linux ARM64 builds):**
```
Error: exec user process caused: exec format error
```
Solution: Run the QEMU registration command: `docker run --rm --privileged multiarch/qemu-user-static --reset -p yes`

**Docker not running:**
```
Error: Cannot connect to the Docker daemon
```
Solution: Start Docker Desktop and ensure it's fully running.

**Slow ARM64 builds:**
ARM64 builds on x86_64 use QEMU emulation and are significantly slower (5-10x) than native builds. This is expected. To speed up:
- Build only necessary Python versions: `export CIBW_BUILD="cp312-*"`
- Use GitHub Actions with native ARM64 runners
- Build on actual ARM64 hardware

**Windows ARM64 build tools missing:**
```
Error: MSVC for ARM64 not found
```
Solution: Install Visual Studio 2022 with:
- "Desktop development with C++" workload
- "MSVC v143 - VS 2022 C++ ARM64 build tools" component

**Out of disk space:**
Building for all platforms/architectures requires significant disk space (~10-20 GB). Clean up old builds:
```bash
rm -rf build/ dist/ wheelhouse/ *.egg-info
```

### Uploading to PyPI

**Test PyPI (recommended first):**
```bash
pip install twine
twine upload --repository testpypi wheelhouse/*
```

**Production PyPI:**
```bash
twine upload wheelhouse/*
```


## Platform-Specific Build Requirements

### Linux

**Requirements:**
- Docker must be running (cibuildwheel uses manylinux containers)
- Boost, CMake, and HDF5 are installed automatically via DNF in the manylinux_2_28 image

**Build Configuration:**
- Image: `manylinux_2_28` (based on AlmaLinux 8)
- Architectures: x86_64, aarch64
- Boost version: System-provided (1.66+)
- Package manager: DNF

**Dependencies installed automatically:**
```bash
dnf install -y boost-devel cmake hdf5-devel
```

**Wheel repair:**
- Uses `auditwheel` to bundle Boost and HDF5 shared libraries into the wheel
- Ensures compatibility across different Linux distributions

**ARM64 Builds:**
- Built using QEMU emulation on x86_64 hosts
- AVX optimizations are automatically disabled on ARM architectures

### Windows

**Requirements:**
- Visual Studio 2017 or later with C++ build tools
- PowerShell (for running build scripts)
- For ARM64: Visual Studio 2022 with ARM64 build tools

**Build Configuration:**
- Boost version: 1.86.0 (built from source)
- Architectures: AMD64, ARM64
- Python versions: 3.8-3.13

**Boost Build Process:**
The Windows build automatically:
1. Downloads Boost 1.86.0 from multiple mirrors (GitHub, SourceForge, JFrog, archives.boost.io)
2. Extracts and caches the archive
3. Detects target architecture (AMD64 or ARM64) from environment
4. Builds only the `locale` component with:
   - Toolset: MSVC
   - Address model: 64-bit
   - Architecture: x86 (AMD64) or arm (ARM64)
   - Variant: Release
   - Link type: Shared (DLLs)
   - Runtime link: Shared (/MD)
5. Installs to `{project}/dist/boost-install-{arch}` (architecture-specific)

**Environment Variables Set:**
```powershell
$env:BOOST_ROOT = "{project}/dist/boost-install-{arch}"
$env:BOOST_INCLUDEDIR = "{project}/dist/boost-install-{arch}/include"
$env:BOOST_LIBRARYDIR = "{project}/dist/boost-install-{arch}/lib"
$env:CMAKE_PREFIX_PATH = "{project}/dist/boost-install-{arch}"
$env:BOOST_ALL_NO_LIB = "1"
```

**Wheel repair:**
- Uses `delvewheel` to bundle Boost DLLs into the wheel
- Command: `delvewheel repair --add-path "{project}\\dist\\boost-install-{arch}\\lib" -w {dest_dir} {wheel}`

**Build Script:**
The PowerShell script `.github/ci/build_boost_windows.ps1` handles:
- Architecture detection (AMD64 vs ARM64)
- Checking for existing Boost installation
- Downloading from multiple mirrors with retry logic
- Extracting with 7-Zip, tar, or Expand-Archive
- Caching extracted files for faster subsequent builds
- Building with b2 (Boost.Build) for the target architecture
- Installing headers and libraries

### macOS

**Requirements:**
- Xcode command line tools
- Boost (can be installed via Homebrew)

**Build Configuration:**
- Deployment target: macOS 10.14+
- Architectures: x86_64, arm64 (Apple Silicon)
- Python versions: 3.8-3.13

**Install Boost:**
```bash
brew install boost
```

**Environment Variables:**
```bash
export CFLAGS="-g -Wall"
export CXXFLAGS="${CXXFLAGS} -D_LIBCPP_DISABLE_AVAILABILITY=1"
export MACOSX_DEPLOYMENT_TARGET="10.14"
```

**Wheel repair:**
- Uses `delocate` to bundle shared libraries
- Ensures compatibility across macOS versions

**Architecture-Specific Notes:**
- x86_64 builds include AVX optimizations
- arm64 builds use standard ARM optimizations (no AVX)
- Universal2 wheels are not currently built (separate wheels for each architecture)


## Compilation from Source

``tttrlib`` can be compiled and installed using the source code provided in the
git repository. The continuous integration pipeline can be used as a reference 
to setup a compilation machine.

```bash
git clone --recursive https://github.com/fluorescence-tools/tttrlib.git
cd tttrlib
python setup.py install
```

### Prerequisites

To compile ``tttrlib`` the following prerequisites need to be fulfilled:

1. An installed C++ compiler (GCC, Clang, or MSVC)
2. The [HDF5](https://www.hdfgroup.org/) library with C/C++ include files
3. A recent 64-bit [Python](https://www.python.org/) installation (3.8+) with include files
4. [CMake](https://cmake.org/) (3.13 or later)
5. [SWIG](http://www.swig.org/) (for generating Python bindings)
6. [Boost](https://www.boost.org/) (1.36 or later, locale component required)

### Debug Build

To debug and analyze the output of ``tttrlib`` in more detail, build the library 
in Debug mode that outputs additional information to the standard logging stream:

```bash
python setup.py build_ext --debug
```

### Conda Build

A conda recipe is provided in the folder 'conda-recipe' to build the tttrlib library with the
[conda build](https://docs.conda.io/projects/conda-build/en/latest/) environment:

```bash
conda install conda-build
conda build build-tools/conda-recipe
```


## Continuous Integration

The project uses GitHub Actions to automatically build wheels for all platforms and architectures. 
See `.github/workflows/` for CI configuration. Wheels are built on:

- **Linux**: manylinux_2_28 (x86_64, aarch64)
- **Windows**: Python 3.8-3.13 (AMD64, ARM64)
- **macOS**: Python 3.8-3.13 (x86_64, arm64)

The CI pipeline:
1. Checks out code with submodules
2. Sets up platform-specific build environment
3. Builds wheels using cibuildwheel for all architectures
4. Tests wheels with basic import and functionality checks
5. Uploads artifacts for distribution

**Cross-compilation:**
- Linux ARM64: Uses QEMU emulation in Docker
- Windows ARM64: Uses Visual Studio cross-compilation tools
- macOS ARM64: Native builds on Apple Silicon runners or cross-compilation


## Troubleshooting

### Boost locale not found

If CMake cannot find the Boost locale component:

**Linux:**
```bash
sudo dnf install boost-devel  # RHEL/CentOS/Fedora
sudo apt install libboost-locale-dev  # Debian/Ubuntu
```

**macOS:**
```bash
brew install boost
```

**Windows:**
The build script should handle this automatically. If it fails, check:
- Internet connectivity for downloading Boost
- Disk space in `dist/` directory
- Visual Studio installation (including ARM64 tools for ARM64 builds)

### CMake version too old

tttrlib requires CMake 3.13 or later. Update CMake:

```bash
pip install --upgrade cmake
```

### SWIG not found

Install SWIG:

**Linux:**
```bash
sudo dnf install swig  # RHEL/CentOS/Fedora
sudo apt install swig  # Debian/Ubuntu
```

**macOS:**
```bash
brew install swig
```

**Windows:**
Download from [swig.org](http://www.swig.org/download.html) and add to PATH.

### Architecture Mismatch

If you encounter errors about architecture mismatches:

**Check your Python architecture:**
```python
import platform
print(platform.machine())  # Should show AMD64, ARM64, x86_64, aarch64, etc.
```

**Ensure you're using the correct wheel:**
- pip should automatically select the right wheel for your architecture
- If building from source, the build system detects architecture automatically